# Short-circuit (lazy) evaluation

**Subject:** Introduction to Problem Solving and Reasoning (IPSR)  
**Unit 1 (Python Basics), Topic 10**  
**How to use this notebook:** read each section, run every code cell (Shift + Enter), then change the examples and run them again. Try each exercise before you open its solution.

## 1. Learning goals

After this notebook you can:

* explain what `and` and `or` really return (an operand, not always `True` or `False`)
* explain what **short-circuit** (lazy) evaluation means and show it with a function that prints
* use guard patterns to avoid errors such as division by zero and `IndexError`
* use `or` to give a default value, and know when this goes wrong (`0` and `""`)
* explain how `any()` and `all()` stop early
* choose between a ternary expression and `or`
* tell `and`/`or` apart from `&`/`|`
* predict the order of evaluation when expressions have side effects

## 2. A quick reminder: truthy and falsy

Every Python value can be used as a condition. These values count as **falsy** (they act like `False`):

* `False`, `None`
* the number zero: `0`, `0.0`
* empty things: `""`, `[]`, `()`, `{}`

Everything else is **truthy**. `bool(x)` shows how Python sees a value.

In [1]:
for value in [0, 1, -5, 0.0, "", "hi", " ", [], [0], None, False]:
    print(repr(value).ljust(6), "->", bool(value))

0      -> False
1      -> True
-5     -> True
0.0    -> False
''     -> False
'hi'   -> True
' '    -> True
[]     -> False
[0]    -> True
None   -> False
False  -> False


Notice that `" "` (a space) and `[0]` (a list holding a zero) are truthy. They are not empty.

## 3. `and` and `or` return an operand

In many languages `a and b` gives only `True` or `False`. In Python it gives **one of the two values**:

* `a and b`: if `a` is falsy, the answer is `a`. Otherwise the answer is `b`.
* `a or b`: if `a` is truthy, the answer is `a`. Otherwise the answer is `b`.

In [2]:
print(True and False)      # False
print(5 and 7)             # 5 is truthy, so we get the second value: 7
print(0 and 7)             # 0 is falsy, so we get 0 straight away
print(5 or 7)              # 5 is truthy, so we get 5
print(0 or 7)              # 0 is falsy, so we get the second value: 7
print("" or "empty")
print("name" and "ok")

False
7
0
5
7
empty
ok


When both sides are real `True`/`False` values, the answer is also `True`/`False`, so you may never notice. The difference
shows when you use numbers, strings or lists as conditions.

`not` is different. It always returns a real `True` or `False`.

In [3]:
print(not 5, not 0, not "", not "x")

False True True False


## 4. What short-circuit means

Python reads `a and b` from left to right and **stops as soon as the answer is known**.

* `a and b`: if `a` is falsy the answer must be falsy, so `b` is **never looked at**.
* `a or b`: if `a` is truthy the answer must be truthy, so `b` is **never looked at**.

This is also called **lazy evaluation**: Python does no more work than needed.

We can see this with a function that prints a message whenever it is called.

In [4]:
def check(label, result):
    print("   check", label, "was called")
    return result

print("Test 1: False and check('A', True)")
print("  answer:", False and check("A", True))

print("Test 2: True and check('B', True)")
print("  answer:", True and check("B", True))

Test 1: False and check('A', True)
  answer: False
Test 2: True and check('B', True)
   check B was called
  answer: True


In Test 1 the message never appears. `check` was **not called** because `False and ...` is already `False`.
Now the same idea for `or`.

In [5]:
print("Test 3: True or check('C', False)")
print("  answer:", True or check("C", False))

print("Test 4: False or check('D', False)")
print("  answer:", False or check("D", False))

Test 3: True or check('C', False)
  answer: True
Test 4: False or check('D', False)
   check D was called
  answer: False


| Expression | Is the right side evaluated? |
|---|---|
| `False and right` | no |
| `True and right` | yes |
| `True or right` | no |
| `False or right` | yes |

## 5. Guard patterns: protect the risky part

The right side of `and` only runs when the left side passed. So the left side can act as a **guard**.

### Guard against division by zero

In [6]:
x = 0
try:
    print(10 / x > 2)
except ZeroDivisionError as e:
    print("Without a guard:", e)

print("With a guard:", x != 0 and 10 / x > 2)    # 10 / x is never computed

for x in [0, 2, 10]:
    print(x, "->", x != 0 and 10 / x > 2)

Without a guard: division by zero
With a guard: False
0 -> False
2 -> True
10 -> False


### Guard against an empty list
`lst[0]` fails on an empty list. `lst and lst[0]` checks first. An empty list is falsy, so the answer is the empty list itself.

In [7]:
marks = []
try:
    print(marks[0])
except IndexError as e:
    print("Without a guard:", e)

print("With a guard:", marks and marks[0])        # gives [] (the falsy value itself)

marks = [72, 65]
print("Non-empty list:", marks and marks[0])      # gives 72

Without a guard: list index out of range
With a guard: []
Non-empty list: 72


### Guard against `None`
A very common case: a value may be `None` (nothing). Check before you use it.

In [8]:
name = None
print(name is not None and name.upper())     # None is not a string, but .upper() is never called

name = "asha"
print(name is not None and name.upper())

False
ASHA


**Order matters.** Put the safe check on the left and the risky one on the right. If you swap them, the guard is useless.

In [9]:
x = 0
try:
    print(10 / x > 2 and x != 0)        # wrong order: the risky part runs first
except ZeroDivisionError as e:
    print("Wrong order:", e)

Wrong order: division by zero


## 6. `or` for default values

Because `a or b` gives `a` when `a` is truthy and `b` otherwise, it is a short way to say "use `a`, or else a default".

In [10]:
user_name = ""
print("Hello,", user_name or "guest")

user_name = "Ravi"
print("Hello,", user_name or "guest")

def total(items=None):
    items = items or []       # if items is None, use an empty list
    return sum(items)

print(total(), total([1, 2, 3]))

Hello, guest
Hello, Ravi
0 6


### Edge case: the trap with `0` and `""`
`or` replaces **every falsy value**, not only "missing" ones. A real `0` or a real empty string gets replaced too.

In [11]:
# A student scored 0 on purpose. We want to keep the 0.
score = 0
print("Using or:   ", score or 100)         # wrong: 0 is thrown away

print("Using None: ", score if score is not None else 100)    # right

# Same for an empty string that is a valid answer
nickname = ""
print(repr(nickname or "no nickname"))

Using or:    100
Using None:  0
'no nickname'


**Rule:** if `0`, `""` or `[]` are valid values for your data, do not use `or` for the default. Test `is None` instead.

## 7. Ternary expression vs `or`

The conditional (ternary) expression is `value_if_true if condition else value_if_false`.
It is more explicit than `or` and has no falsy trap, because **you choose the condition**.

In [12]:
age = 17
print("adult" if age >= 18 else "minor")

score = 0
print(score if score is not None else 100)     # keeps the 0
print(score or 100)                            # replaces the 0

# Only the chosen branch is evaluated, so this is also lazy
x = 0
print(10 / x if x != 0 else "cannot divide by zero")

minor
0
100
cannot divide by zero


Use `or` for a quick default when every falsy value really means "missing". Use the ternary expression when you need to be precise.

## 8. `any()` and `all()` also stop early

* `any(items)` is `True` as soon as it finds one truthy item.
* `all(items)` is `False` as soon as it finds one falsy item.

When you give them a **generator** (a lazy list), they stop asking for items at that point.

In [13]:
print(any([0, "", 3, None]))      # True
print(all([1, "a", [0]]))         # True
print(any([]), all([]))           # False True  (empty: nothing to find, nothing to fail)

True
True
False True


In [14]:
def is_even(n):
    print("   testing", n)
    return n % 2 == 0

numbers = [3, 5, 8, 9, 11, 13]
print("any: stops at the first even number")
print(any(is_even(n) for n in numbers))      # tests 3, 5, 8 then stops

print("all: stops at the first failure")
print(all(is_even(n) for n in numbers))      # tests 3 then stops

any: stops at the first even number
   testing 3
   testing 5
   testing 8
True
all: stops at the first failure
   testing 3
False


The first even number is 8, so 9, 11 and 13 are never tested. Compare with a list comprehension in square brackets,
which builds the full list first and does **not** stop early.

In [15]:
print(any([is_even(n) for n in [3, 8, 9]]))   # tests all three, then any() looks at the list

   testing 3
   testing 8
   testing 9
True


## 9. `&` and `|` are not `and` and `or`

| | `and` / `or` | `&` / `|` |
|---|---|---|
| Name | logical operators | bitwise operators |
| Short-circuit | yes | **no** (both sides always run) |
| Returns | one of the operands | a new value built bit by bit |
| Use for | conditions | integer bits, sets, arrays |

In [16]:
def noisy(label, value):
    print("   evaluating", label)
    return value

print("with 'and':")
print(False and noisy("right side", True))

print("with '&':")
print(False & noisy("right side", True))      # right side runs anyway

with 'and':
False
with '&':
   evaluating right side
False


Another difference: `&` and `|` bind **tighter** than comparison operators, so brackets are needed.

In [17]:
x = 5
print(x > 2 and x < 10)       # normal use
print((x > 2) & (x < 10))     # works, but only with the brackets
print(6 & 3, 6 | 3)           # on integers they work on bits: 110 & 011 = 010, 110 | 011 = 111

True
True
2 7


For plain conditions in your programs, always use `and`, `or`, `not`.

## 10. Evaluation order and side effects

A **side effect** is something a function does besides returning a value, for example printing or changing a list.
Python evaluates `and`/`or` from left to right, so side effects follow that order, and skipped parts have **no** side effect.

In [18]:
log = []

def step(name, result):
    log.append(name)
    return result

step("A", True) and step("B", False) and step("C", True)
print(log)          # C was skipped because B was False

log.clear()
step("A", False) or step("B", True) or step("C", True)
print(log)          # C was skipped because B was True

['A', 'B']
['A', 'B']


Precedence: `not` first, then `and`, then `or`. So `a or b and c` means `a or (b and c)`.

In [19]:
print(True or False and False)       # True  : and first -> True or False
print((True or False) and False)     # False : brackets change the meaning

True
False


Avoid putting important side effects (such as `list.append`, `input()` or `print`) inside conditions. Another reader may not
notice that they might be skipped.

## 11. Edge cases

Collect the tricky behaviour in one place.

In [20]:
# 1. The result may not be a bool
print(repr([] or 0), repr([] and 0), repr("a" and 0))

# 2. A chain returns the first falsy value (and) or the first truthy value (or)
print(1 and 2 and 0 and 3)       # 0
print(0 or "" or None or "x")    # x
print(0 or "" or None)           # None : last operand is returned when none are truthy

# 3. Chained comparisons also stop at the first failed comparison
def val(n):
    print("   val", n)
    return n
print(1 > val(2) < val(3))       # first comparison fails, so val(3) is never called

# 4. NaN is truthy
nan = float("nan")
print(bool(nan))

# 5. A function object is always truthy: forgetting the brackets is a classic bug
def is_ready():
    return False
print("bug:", bool(is_ready))        # True, because it is the function itself
print("fixed:", bool(is_ready()))

0 [] 0
0
x
None
   val 2
False
True
bug: True
fixed: False


## 12. Quick summary

| Idea | Rule |
|---|---|
| `a and b` | `a` if `a` is falsy, else `b` |
| `a or b` | `a` if `a` is truthy, else `b` |
| Short-circuit | the right side is skipped when the left side decides the answer |
| Guard | `x != 0 and 10 / x > 2`, `lst and lst[0]` |
| Default with `or` | fine, but `0`, `""` and `[]` are replaced too |
| Precise default | `a if a is not None else default` |
| `any` / `all` | stop at the first deciding item (with a generator) |
| `&` / `|` | bitwise, **never** short-circuit, need brackets |
| Precedence | `not`, then `and`, then `or` |

## 13. Exercises

### Exercise 1

Predict each result, then run to check: `0 or 5`, `3 and 0`, `"" or []`, `"a" and "b" and "c"`, `None or 0 or ""`.

*Hint:* and returns the first falsy value, or the last value. or returns the first truthy value, or the last value.

In [21]:
# Write your answer here

<details>
<summary><b>Show solution 1</b></summary>

Results: `5`, `0`, `[]`, `c`, `''`. When no operand decides early, the last operand is the answer.

```python
print(0 or 5)
print(3 and 0)
print(repr("" or []))
print("a" and "b" and "c")
print(repr(None or 0 or ""))
```

</details>

### Exercise 2

Write a function `safe_ratio(a, b)` that returns `True` if `a / b` is greater than 1, and `False` if `b` is zero. Do it in one line with a guard. Test with `(10, 2)`, `(1, 4)` and `(5, 0)`.

In [22]:
# Write your answer here

<details>
<summary><b>Show solution 2</b></summary>

When `b` is 0 the left side is `False`, so `a / b` is never computed. The function returns `False`.

```python
def safe_ratio(a, b):
    return b != 0 and a / b > 1

print(safe_ratio(10, 2), safe_ratio(1, 4), safe_ratio(5, 0))
```

</details>

### Exercise 3

Write `first_or_default(lst, default)` that returns the first item of a list, or `default` when the list is empty. It must also work when the first item is `0`. Test with `[0, 1]`, `[]` and `["x"]`.

*Hint:* `lst[0] if lst else default` is safe. Why is `(lst and lst[0]) or default` wrong for `[0, 1]`?

In [23]:
# Write your answer here

<details>
<summary><b>Show solution 3</b></summary>

The ternary form checks the list, not the first item, so a first item of `0` is kept. With `(lst and lst[0]) or default` the `0` would be falsy and replaced by the default.

```python
def first_or_default(lst, default):
    return lst[0] if lst else default

print(first_or_default([0, 1], "none"))
print(first_or_default([], "none"))
print(first_or_default(["x"], "none"))
```

</details>

### Exercise 4

Make a function `greet(name=None)` that prints `Hello, guest` when no name is given and `Hello, <name>` otherwise. Use `or`. Then explain in a comment what happens when `name` is the empty string.

In [24]:
# Write your answer here

<details>
<summary><b>Show solution 4</b></summary>

For names, treating an empty string like a missing name is usually what we want, so `or` is a good fit here.

```python
def greet(name=None):
    print("Hello,", name or "guest")

greet()
greet("Meera")
greet("")      # empty string is falsy, so we also get "guest"
```

</details>

### Exercise 5

Use `any()` with a generator to check if a list of words has a word longer than 5 letters. Add a print inside a helper function to show that it stops at the first long word. Use `words = ["cat", "elephant", "dog", "giraffe"]`.

In [25]:
# Write your answer here

<details>
<summary><b>Show solution 5</b></summary>

It checks `cat` and `elephant`, finds `elephant` is long, and stops. `dog` and `giraffe` are never checked.

```python
def is_long(word):
    print("   checking", word)
    return len(word) > 5

words = ["cat", "elephant", "dog", "giraffe"]
print(any(is_long(w) for w in words))
```

</details>

### Exercise 6

Run this and explain the two outputs. Which one calls `show`?

```python
def show():
    print("show called")
    return True

print(False and show())
print(False & show())
```

In [26]:
# Write your answer here

<details>
<summary><b>Show solution 6</b></summary>

`and` short-circuits, so `show` is not called and `False` is printed. `&` always evaluates both sides, so `show called` appears first, then `False`.

```python
def show():
    print("show called")
    return True

print(False and show())
print(False & show())
```

</details>

### Exercise 7

A shop stores the discount as a number. A value of `0` means a real 0 percent discount, and `None` means 'not set, use 10'. Write `get_discount(d)` that works correctly for `0`, `None` and `25`. Show why `d or 10` is wrong.

In [27]:
# Write your answer here

<details>
<summary><b>Show solution 7</b></summary>

`d or 10` turns a real `0` into `10`. Testing `is not None` replaces only the missing value.

```python
def get_discount(d):
    return d if d is not None else 10

print(get_discount(0), get_discount(None), get_discount(25))

d = 0
print("wrong version gives", d or 10)
```

</details>

## 14. Where to read more

* Real Python, *Python Booleans* (and/or and truthiness): https://realpython.com/python-boolean/
* Python Morsels, *Short-circuit evaluation* (preview of a paid screencast): https://www.pythonmorsels.com/short-circuit-evaluation/